In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import matplotlib.pyplot as plt

import random

#Load the sidewalk GeoJSON
sidewalks = gpd.read_file("Sidewalk_Centerline.geojson")

#Load the streetlight CSV as a GeoDataFrame
streetlights_df = pd.read_csv("streetlight-locations.csv")

# Assuming columns are 'lon' and 'lat'
streetlights = gpd.GeoDataFrame(
    streetlights_df,
    geometry=gpd.points_from_xy(streetlights_df.Long, streetlights_df.Lat),
    crs="EPSG:4326"  # change if your data uses another CRS
)

# Reproject to a planar coordinate system (for accurate distance)
# You can use a local UTM zone; here’s a generic example:
sidewalks = sidewalks.to_crs(epsg=3857)
streetlights = streetlights.to_crs(epsg=3857)

# Spatial join: find which streetlights are within a certain distance of each sidewalk
# Define how close a light must be to count (e.g., 5 meters)
buffer_distance = 5

# Create sidewalk buffers
sidewalk_buffers = sidewalks.copy()
sidewalk_buffers["geometry"] = sidewalks.buffer(buffer_distance)

# Perform spatial join: which lights fall in which buffer
joined = gpd.sjoin(streetlights, sidewalk_buffers, predicate="within")

# Count streetlights per sidewalk
counts = joined.groupby("index_right").size()

# Add count back to the sidewalk GeoDataFrame
sidewalks["num_lights"] = sidewalks.index.map(counts).fillna(0).astype(int)

# Save result
sidewalks.to_file("sidewalks_with_lights.geojson", driver="GeoJSON")

# Load your GeoJSON
sidewalks = gpd.read_file("sidewalks_with_lights.geojson")

In [ ]:
user_start = (42.3505, -71.1087)   # Marsh Plaza
user_dest  = (42.3605, -71.0591)   # BU FitRec (Fitness & Recreation Center)

# Create GeoDataFrames for both points
user_point = gpd.GeoDataFrame(
    geometry=[Point(user_start[1], user_start[0])],
    crs="EPSG:4326"
).to_crs(sidewalks.crs)

dest_point = gpd.GeoDataFrame(
    geometry=[Point(user_dest[1], user_dest[0])],
    crs="EPSG:4326"
).to_crs(sidewalks.crs)

# Find the nearest sidewalks to start and end
nearest_start_idx = sidewalks.distance(user_point.iloc[0].geometry).idxmin()
nearest_end_idx   = sidewalks.distance(dest_point.iloc[0].geometry).idxmin()

print("Start sidewalk index:", nearest_start_idx)
print("End sidewalk index:", nearest_end_idx)

Start sidewalk index: 87815
End sidewalk index: 103904


In [ ]:
import folium
import networkx as nx
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points

# 1. Hardcode start & dest
user_start = (42.3505, -71.1087)   # Marsh Plaza
user_dest  = (42.34543, -71.10612)   # BU Peabody Hall

start_gdf = gpd.GeoDataFrame(
    geometry=[Point(user_start[1], user_start[0])],
    crs="EPSG:4326"
).to_crs(sidewalks.crs)

dest_gdf = gpd.GeoDataFrame(
    geometry=[Point(user_dest[1], user_dest[0])],
    crs="EPSG:4326"
).to_crs(sidewalks.crs)

start_pt = start_gdf.iloc[0].geometry
end_pt   = dest_gdf.iloc[0].geometry


# 2. Build a graph from sidewalks
G = nx.Graph()

# we assume each row in `sidewalks` has a LineString geometry
for idx, row in sidewalks.iterrows():
    geom = row.geometry

    # skip weird rows
    if geom is None or geom.is_empty:
        continue

    # If it's a MultiLineString, break it up
    if geom.geom_type == "MultiLineString":
        lines = list(geom.geoms)
    else:
        lines = [geom]

    for line in lines:
        # get all coordinates along the line
        coords = list(line.coords)

        # add edges between consecutive coords
        # so a long curvy sidewalk becomes many small edges
        for i in range(len(coords) - 1):
            a = coords[i]
            b = coords[i+1]

            # use the actual metric distance between the two points
            pa = Point(a)
            pb = Point(b)
            dist = pa.distance(pb)

            # add to graph
            G.add_node(a, pos=a)
            G.add_node(b, pos=b)
            G.add_edge(a, b, weight=dist, geometry=LineString([a, b]))


# 3. Helper: find nearest graph node to an arbitrary point
# We’ll project each node to a Shapely Point and get the closest.
# This is O(N). For city-scale data it's usually fine.

def nearest_graph_node(point_geom, graph):
    min_node = None
    min_dist = float("inf")
    for n, data in graph.nodes(data=True):
        node_point = Point(n)
        d = point_geom.distance(node_point)
        if d < min_dist:
            min_dist = d
            min_node = n
    return min_node

start_node = nearest_graph_node(start_pt, G)
end_node   = nearest_graph_node(end_pt, G)

print("Start node:", start_node)
print("End node:", end_node)

In [ ]:



# 4. Compute shortest paths
# We'll ask for the first 3 different simple paths ranked by total weight.
# shortest_simple_paths yields paths in increasing total weight.

def path_length(graph, path):
    total = 0.0
    for i in range(len(path) - 1):
        total += graph[path[i]][path[i+1]]["weight"]
    return total

k = 3
all_paths_generator = nx.shortest_simple_paths(G, start_node, end_node, weight="weight")

top_paths = []
for p in all_paths_generator:
    top_paths.append(p)
    if len(top_paths) == k:
        break

# 5. Convert each path (list of nodes) into a LineString for mapping
route_geoms = []
route_lengths_m = []

for p in top_paths:
    line = LineString(p)  # p is list of (x,y) tuples already in projected CRS
    route_geoms.append(line)
    route_lengths_m.append(path_length(G, p))

# Make a GeoDataFrame of the routes
routes_gdf = gpd.GeoDataFrame(
    {"rank": list(range(1, len(route_geoms)+1)),
     "length_m": route_lengths_m},
    geometry=route_geoms,
    crs=sidewalks.crs
)

print(routes_gdf[["rank", "length_m"]])

# 6. (Optional) visualize in Folium
# convert everything back to WGS84 for web map
routes_display = routes_gdf.to_crs(epsg=4326)
start_display = start_gdf.to_crs(epsg=4326)
end_display = dest_gdf.to_crs(epsg=4326)
sidewalks_display = sidewalks.to_crs(epsg=4326)

m = folium.Map(
    location=[start_display.geometry.y.mean(), start_display.geometry.x.mean()],
    zoom_start=15
)

# sidewalks (light gray, thin)
folium.GeoJson(
    sidewalks_display,
    name="Sidewalks",
    style_function=lambda x: {
        "color": "#888888",
        "weight": 1,
        "opacity": 0.5
    }
).add_to(m)

# top 3 routes, thicker
colors = ["red", "blue", "green"]
for i, row in routes_display.iterrows():
    folium.GeoJson(
        row.geometry,
        name=f"Route {row['rank']} ({round(row['length_m'],1)} m)",
        style_function=lambda x, col=colors[i % len(colors)]: {
            "color": col,
            "weight": 5,
            "opacity": 0.8
        }
    ).add_to(m)

# start marker
folium.Marker(
    location=[start_display.geometry.y.iloc[0], start_display.geometry.x.iloc[0]],
    popup="START",
    icon=folium.Icon(color="green")
).add_to(m)

# end marker
folium.Marker(
    location=[end_display.geometry.y.iloc[0], end_display.geometry.x.iloc[0]],
    popup="DEST",
    icon=folium.Icon(color="red")
).add_to(m)

m